In [1]:
!pip install -q langchain-openai langchain-community langchain-core python-dotenv
print("Librerías instaladas.")

Librerías instaladas.


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.3
)

if os.getenv("GITHUB_TOKEN"):
    print(f"Conectado exitosamente. Token: {os.getenv('GITHUB_TOKEN')[:4]}...")
else:
    print("Error: No se encontró GITHUB_TOKEN")

Conectado exitosamente. Token: gith...


In [3]:
# Celda 2: Configuración Zero-Shot + Memoria Manual
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# --- CONFIGURACIÓN DEL PROMPT ZERO-SHOT ---
# No incluimos ejemplos (Few-Shot), solo el ROL y las REGLAS.
PROMPT_SISTEMA = """
Eres un Coordinador de Experiencia Estudiantil en DuocUC. 
Tu tarea es clasificar y resolver dudas de alumnos de forma profesional.

### INSTRUCCIONES:
1. Analiza el sentimiento del mensaje (Positivo, Neutral, Negativo).
2. Clasifica el ticket en uno de estos departamentos: [TI, Académico, Finanzas, Biblioteca].
3. Proporciona una respuesta breve y empática.

### REGLA DE MEMORIA:
- Revisa el historial de la conversación. 
- Si el alumno ya mencionó su nombre, carrera o un problema previo, personaliza tu respuesta basándote en esos datos.
- NO pidas información que el alumno ya entregó anteriormente.

### FORMATO DE SALIDA:
SENTIMIENTO: [Sentimiento]
DEPARTAMENTO: [Departamento]
RESPUESTA: [Tu mensaje al alumno]
"""

# Mantenemos la lógica de memoria manual que te gustó
historial_mensajes = []

def procesador_zero_shot(texto_usuario):
    mensajes_para_enviar = [SystemMessage(content=PROMPT_SISTEMA)]
    
    # Inyectamos la memoria acumulada
    for msg in historial_mensajes:
        mensajes_para_enviar.append(msg)
    
    nuevo_mensaje = HumanMessage(content=texto_usuario)
    mensajes_para_enviar.append(nuevo_mensaje)
    
    respuesta = llm.invoke(mensajes_para_enviar)
    
    # Guardamos en el historial
    historial_mensajes.append(nuevo_mensaje)
    historial_mensajes.append(AIMessage(content=respuesta.content))
    
    return respuesta.content

print("✓ Configuración Zero-Shot completada.")
print("✓ El modelo ahora depende 100% de la claridad de las instrucciones.")

✓ Configuración Zero-Shot completada.
✓ El modelo ahora depende 100% de la claridad de las instrucciones.


In [4]:
# Reiniciamos para la prueba
historial_mensajes = []

# Interacción 1: El alumno se presenta
print("--- TEST 1 ---")
print(procesador_zero_shot("Hola, soy Ignacio Salazar y no puedo entrar a mi correo Duoc."))

# Interacción 2: El alumno hace una pregunta de seguimiento
print("\n--- TEST 2 ---")
print(procesador_zero_shot("¿A qué departamento mandaste mi caso y cómo te dije que me llamaba?"))

--- TEST 1 ---
SENTIMIENTO: Negativo  
DEPARTAMENTO: TI  
RESPUESTA: Hola Ignacio, lamento que estés teniendo problemas con tu correo Duoc. Te recomiendo intentar restablecer tu contraseña desde la opción "¿Olvidaste tu contraseña?" en la página de acceso. Si el problema persiste, por favor contáctanos nuevamente para ayudarte a resolverlo.

--- TEST 2 ---
SENTIMIENTO: Neutral  
DEPARTAMENTO: TI  
RESPUESTA: Hola Ignacio, mandé tu caso al departamento de TI, ya que mencionaste que no podías acceder a tu correo Duoc. También recuerdo que me dijiste que tu nombre es Ignacio Salazar. Si necesitas más ayuda, no dudes en decírmelo. ¡Estoy aquí para apoyarte!
